# Piso 3ºB · Capítulo → audio con Chatterbox España (en tu GPU, sin cuotas)
**Cada día:** pega el capítulo en la celda 5 → *Entorno de ejecución → Ejecutar todo* → se descarga `audio_capitulo.json` → en la web, «Audio de Colab» → «Crear vídeo».

**Primera vez:** *Entorno de ejecución → Cambiar tipo → GPU T4*. La celda 2 reinicia la sesión una vez (es normal): cuando pase, pulsa otra vez *Ejecutar todo*.
La celda 4 genera una **prueba numerada** de los 6 personajes para que me digas cuáles te gustan.

In [ ]:
# 1) Librerías (la primera vez reinicia la sesión una vez; después ya no)
import os, subprocess
if not os.path.exists('/content/.chatterbox_listo'):
    subprocess.run("pip install -q resampy librosa s3tokenizer omegaconf resemble-perth silero-vad conformer safetensors diffusers==0.29.0 transformers==4.46.3 edge-tts", shell=True)
    open('/content/.chatterbox_listo', 'w').write('1')
    print("Librerías instaladas. Reiniciando la sesión... vuelve a pulsar «Ejecutar todo».")
    os.kill(os.getpid(), 9)
print("Librerías listas.")

In [ ]:
# 2) Descargar el código de la demo de Chatterbox España y cargar el modelo en la GPU
import sys, os, subprocess, random, numpy as np, torch, re, json, base64, shutil
if not os.path.exists('/content/sp'):
    subprocess.run("git clone -q https://huggingface.co/spaces/ResembleAI/Chatterbox-Multilingual-TTS-es-es /content/sp", shell=True, check=True)
sys.path.insert(0, '/content/sp'); os.chdir('/content/sp')
from chatterbox.src.chatterbox.tts import ChatterboxTTS
model = ChatterboxTTS.from_pretrained("cuda")
SR = model.sr; print("Modelo cargado. Frecuencia:", SR)
def seed(n):
    random.seed(n); np.random.seed(n); torch.manual_seed(n); torch.cuda.manual_seed_all(n)
os.makedirs('/content/refs', exist_ok=True); os.makedirs('/content/salida', exist_ok=True)

In [ ]:
# 3) Voces de referencia: las MUESTRAS OFICIALES de España de la demo (hablantes nativos, hombre y mujer), y variantes de tono para los personajes
import requests, librosa, soundfile as sf
from IPython.display import Audio, display, HTML
B1 = "https://storage.googleapis.com/chatterbox-demo-samples/mtl-v3-single-language-prompts/es-es/"
B2 = "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/"
cand = [B1 + f"es_es_{g}{n}.{ext}" for g in "fm" for n in (1, 2, 3, 4) for ext in ("wav", "flac")] + [B2 + f"es_{g}{n}.{ext}" for g in "fm" for n in (1, 2, 3) for ext in ("flac", "wav")]
base = {}   # nombre -> ruta local
for u in cand:
    nom = u.split("/")[-1].rsplit(".", 1)[0]
    if nom in base: continue
    try:
        r = requests.get(u, timeout=20)
        if r.status_code == 200 and len(r.content) > 5000:
            p = "/content/refs/" + u.split("/")[-1]; open(p, "wb").write(r.content); base[nom] = p
    except Exception as e: pass
print("Muestras de España encontradas:", list(base))
for nom, p in base.items():
    display(HTML(f"<b>{nom}</b>")); display(Audio(p))
fem = [n for n in base if re.search(r"_f\d", n) or n.startswith("es_f")]; mas = [n for n in base if re.search(r"_m\d", n) or n.startswith("es_m")]
print("Mujeres:", fem, "| Hombres:", mas)
assert fem and mas, "No he podido descargar muestras de hombre y de mujer. Pégame esta salida."

def variante(nom, pasos, salida):
    y, sr0 = librosa.load(base[nom], sr=24000, mono=True)
    if pasos: y = librosa.effects.pitch_shift(y, sr=24000, n_steps=pasos)
    sf.write(f"/content/refs/{salida}.wav", y, 24000)
# personaje: (muestra base, semitonos de cambio de tono, semilla)
M1, M2 = mas[0], mas[-1]; F1, F2 = fem[0], fem[-1]
# madre: si hay una tercera voz de mujer la usa; si no, la de Marta un poco más aguda (bajar el tono la volvía masculina)
MADRE = (fem[1], 0, 61) if len(fem) >= 3 else (F1, 2, 61)
PERS = {
 "yo":     (M1, 0,  11),    # narrador (hombre)
 "amigo":  (M2, 2,  21),    # Dani: hombre joven, algo más agudo
 "amiga":  (F1, 0,  31),    # Marta: la voz de mujer de la demo (la que te gustó)
 "jefe":   (M1, -4, 41),    # casero: más grave
 "abuela": (F2, 1,  51),    # vecina
 "madre":  MADRE,
}
for k, (b, st, sd) in PERS.items(): variante(b, st, k)

ANIMO = {"neutro": (.6, .5), "serio": (.5, .5), "hablar": (.6, .5), "duda": (.55, .5), "cansado": (.4, .3), "triste": (.4, .3), "dormido": (.35, .3),
         "feliz": (.85, .5), "sonrisa": (.75, .5), "risa": (.9, .5), "enfado": (1.0, .5), "asco": (.8, .5), "sorpresa": (1.0, .5), "panic": (1.2, .6)}

def trocear(t, mx=290):
    fr = re.split(r"(?<=[.!?…])\s+", t.strip()); out = []; cur = ""
    for f in fr:
        if len(cur) + len(f) + 1 <= mx: cur = (cur + " " + f).strip()
        else:
            if cur: out.append(cur)
            cur = f
            while len(cur) > mx: out.append(cur[:mx]); cur = cur[mx:]
    if cur: out.append(cur)
    return out or [t[:mx]]
def recortar(w):
    idx = np.where(np.abs(w) > 0.012)[0]
    return w if len(idx) == 0 else w[max(0, idx[0] - int(SR*.03)): min(len(w), idx[-1] + int(SR*.06))]
def hablar(texto, quien, animo="neutro"):
    k = quien if quien in PERS else "yo"; ex, cfg = ANIMO.get(animo, ANIMO["neutro"]); piezas = []
    for ch in trocear(re.sub(r"[*_#]", "", texto)):
        seed(PERS[k][2])
        wav = model.generate(ch, exaggeration=ex, temperature=0.8, cfg_weight=cfg, language_id="es", audio_prompt_path=f"/content/refs/{k}.wav")
        piezas.append(recortar(wav.squeeze(0).cpu().numpy().astype(np.float32)))
    gap = np.zeros(int(SR * .1), dtype=np.float32); sal = []
    for i, p in enumerate(piezas):
        sal.append(p)
        if i < len(piezas) - 1: sal.append(gap)
    return np.concatenate(sal)

In [ ]:
# 4) PRUEBA numerada de los personajes (3 niveles de emoción cada uno). Dime los números que te gustan.
import soundfile as sf
from IPython.display import Audio, display, HTML
FRASE = {
 "yo":     "Son las ocho y ocho, y ya llego tarde otra vez.",
 "amigo":  "Tranquilo, tío, que sobra tiempo. ¿Quién ha dejado la sartén en el suelo?",
 "amiga":  "No pasa nada, es puntualidad. Pero la nevera ha hecho un ruido muy raro.",
 "jefe":   "Esto es un piso, no un hotel. Quiero el alquiler antes del lunes, ¿vale?",
 "abuela": "Ay, hijo, qué ruido hacéis. En mis tiempos esto no pasaba.",
 "madre":  "Cariño, ¿has comido algo? Que te veo muy delgado, de verdad.",
}
n = 0
for k in PERS:
    for animo, nombre in [("cansado", "cansado"), ("neutro", "normal"), ("enfado", "enfadado")]:
        n += 1; w = hablar(FRASE[k], k, animo); f = f"/content/salida/{n:02d}_{k}_{nombre}.wav"; sf.write(f, w, SR)
        display(HTML(f"<div style='font:700 28px sans-serif;margin-top:14px'>Nº {n:02d} · {k} · {nombre}</div>")); display(Audio(f))

In [ ]:
# 5) PEGA AQUÍ EL CAPÍTULO de ChatGPT (entre las tres comillas). Vale con o sin ```json
CAPITULO = r'''

'''

In [ ]:
# 6) Generar el audio de cada escena y descargar audio_capitulo.json
txt = CAPITULO.strip()
if len(txt) < 50: raise SystemExit("Pega primero el capítulo en la celda 5.")
cap = json.loads(txt[txt.index("{"): txt.rindex("}") + 1]); escenas = cap["escenas"][:40]; print(len(escenas), "escenas")
def trozos(e):
    t = re.sub(r"\s+", " ", e["texto"]).strip(); habla = e.get("habla") or "yo"; out = []
    for m in re.finditer(r"«([^»]*)»|([^«»]+)", t):
        if m.group(1) is not None:
            if m.group(1).strip(): out.append((habla, m.group(1).strip()))
        elif re.search(r"[\wÁÉÍÓÚáéíóúÑñ]", m.group(2)): out.append(("yo", re.sub(r"^[\s.,;:!?]+", "", m.group(2)).strip()))
    return out or [("yo", t)]
def animo_de(e, quien):
    for p in ((e.get("plano") or {}).get("pers") or []):
        if p.get("quien") == quien: return p.get("mood") if p.get("mood") in ANIMO else "neutro"
    return "neutro"
audios = []; gap = np.zeros(int(SR * .12), dtype=np.float32)
for i, e in enumerate(escenas):
    ps = [hablar(t, q, animo_de(e, q)) for q, t in trozos(e)]; sal = []
    for j, p in enumerate(ps):
        sal.append(p)
        if j < len(ps) - 1: sal.append(gap)
    audios.append(np.concatenate(sal)); print(f"escena {i+1}/{len(escenas)} · {len(audios[-1])/SR:.1f} s")
def b64(w):
    pk = np.max(np.abs(w)) or 1.0
    if pk > .95: w = w * (.95 / pk)
    return base64.b64encode((np.clip(w, -1, 1) * 32767).astype("<i2").tobytes()).decode()
json.dump({"v": 1, "fs": int(SR), "escenas": [b64(w) for w in audios]}, open("/content/audio_capitulo.json", "w"))
from google.colab import files; files.download("/content/audio_capitulo.json")
print("Total:", round(sum(len(a) for a in audios) / SR), "s")